# 3.1 — One ticket, four patterns
**Session 3.1 · Demo**

What you'll see: the same help-desk tickets handled four ways, from the simplest to the most
structured: a single call, a prompt chain, a router and parallel calls. Each step shows its calls,
tokens, cost and time, and a scorecard at the end compares them.

Each step runs on its own: after the first three steps, you can run or skip any of them.

In [1]:
# @title Choose the model { display-mode: "form" }
MODEL_CHOICE = "Open-weight model"  # @param ["Claude", "Open-weight model"]
OPEN_MODEL = "qwen3:8b"  # @param ["qwen3:8b", "qwen3:4b", "gpt-oss:20b", "llama3.1:8b"]

import importlib.util
import os
import shutil
import subprocess
import sys
import time

import requests
from IPython.display import HTML, display

OLLAMA = "http://127.0.0.1:11434"


def ollama_running():
    try:
        return requests.get(f"{OLLAMA}/api/tags", timeout=5).ok
    except requests.RequestException:
        return False


def start_ollama():
    """Install Ollama if needed, start it, and download the model."""
    if shutil.which("ollama") is None:
        subprocess.run("apt-get install -y -qq pciutils zstd > /dev/null", shell=True)
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True)
    if not ollama_running():
        settings = {"OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_CONTEXT_LENGTH": "16384", "OLLAMA_NUM_PARALLEL": "2"}
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                         env={**os.environ, **settings})
        for _ in range(60):
            if ollama_running():
                break
            time.sleep(1)
    requests.post(f"{OLLAMA}/api/pull", json={"model": OPEN_MODEL, "stream": False}, timeout=1800).raise_for_status()


def without_cache_control(value):
    """Ollama has no prompt caching, so the caching marks are removed."""
    if isinstance(value, dict):
        return {key: without_cache_control(item) for key, item in value.items() if key != "cache_control"}
    if isinstance(value, list):
        return [without_cache_control(item) for item in value]
    return value


def zero_costs():
    """Answers from the local model cost nothing, so the prices the notebook uses are set to zero."""
    kit_llm = sys.modules.get("helpdesk_kit.llm")
    if kit_llm is not None:
        kit_llm.cost = lambda usage, model: 0.0
    if "PRICES" in globals():
        globals()["PRICES"] = (0, 0)


def open_model_create(self, **request):
    """Send a request written for Claude to the open-weight model."""
    request = without_cache_control({**request, "model": OPEN_MODEL})
    request.setdefault("thinking", {"type": "disabled"})  # Thinking is slow on a T4; on only when asked for.
    zero_costs()
    return Messages.claude_create(self, **request)


if MODEL_CHOICE == "Open-weight model":
    if shutil.which("nvidia-smi") is None:
        raise RuntimeError("No GPU: choose Runtime > Change runtime type > T4 GPU, then run the notebook again.")
    start_ollama()
    if importlib.util.find_spec("anthropic") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "anthropic"])
    from anthropic.resources.messages import Messages

    # Every connection made from here on goes to Ollama, which doesn't check the key.
    os.environ.update(ANTHROPIC_BASE_URL=OLLAMA, ANTHROPIC_API_KEY="ollama")
    if not hasattr(Messages, "claude_create"):
        Messages.claude_create = Messages.create
    Messages.create = open_model_create
    message = f"Every request goes to <b>{OPEN_MODEL}</b>."
elif os.environ.get("ANTHROPIC_BASE_URL") == OLLAMA:
    message = "Still on the open-weight model. To use Claude, restart the session and run the notebook again."
else:
    message = "Every request goes to Claude."
USING_OPEN_MODEL = os.environ.get("ANTHROPIC_BASE_URL") == OLLAMA
display(HTML(f'<div style="padding:8px 12px;border-left:4px solid #2a9d8f;background:#f4f8f7">{message}</div>'))

In [2]:
# @title Setup: connect to the model { display-mode: "form" }
import importlib.util, shutil, subprocess, sys

if importlib.util.find_spec("anthropic") is None:
    # uv comes with Colab and installs much faster than pip; pip is the fallback.
    installer = (
        ["uv", "pip", "install", "--python", sys.executable]
        if shutil.which("uv")
        else [sys.executable, "-m", "pip", "install"]
    )
    subprocess.check_call([*installer, "-q", "anthropic"])

import html
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
from getpass import getpass
from threading import Lock

import anthropic
import pandas as pd
from IPython.display import HTML, display

pd.set_option("display.max_colwidth", None)


def read_api_key():
    """Environment variable, then the Colab secret, then a prompt. The key is never printed."""
    key = os.environ.get("ANTHROPIC_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata

        return userdata.get("ANTHROPIC_API_KEY")
    except Exception:  # Not on Colab, or the secret isn't shared with this notebook.
        return getpass("Paste your Anthropic API key: ")


client = anthropic.Anthropic(api_key=read_api_key())
if USING_OPEN_MODEL:
    MODEL, PRICES = OPEN_MODEL, (0, 0)  # Runs in this Colab, with no fee.
else:
    MODEL = "claude-haiku-5-5"  # The small model: every pattern here uses it.
    PRICES = (0.10, 0.50)  # Haiku: dollars per million tokens, (input, output).

# Display helpers, so the results read well on a shared screen.
INK, SOFT_INK, MUTED = "#0b0b0b", "#52514e", "#898781"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"


def text_html(text):
    """Plain text, safe to show as HTML, with its line breaks kept."""
    return f'<div style="white-space:pre-wrap;color:{INK}">{html.escape(str(text))}</div>'


def badge(text, colour=BLUE):
    """A small coloured label, e.g. a category or a route."""
    return (f'<span style="display:inline-block;padding:2px 9px;border-radius:10px;background:{colour};'
            f'color:#fff;font-size:12px;font-weight:600">{html.escape(str(text))}</span>')


def verdict(passed):
    """A pass / fail label: aqua for pass, orange for fail."""
    return badge("pass", AQUA) if passed else badge("fail", ORANGE)


def card(body, label="", colour=BLUE):
    """A white box with a coloured left border and an optional small label on top."""
    label_html = f'<div style="font-size:12px;color:{SOFT_INK};margin-bottom:4px">{label}</div>' if label else ""
    display(HTML(f'<div style="margin:8px 0;padding:10px 14px;border-left:3px solid {colour};'
                 f'background:#fff">{label_html}{body}</div>'))


def caption(text):
    """A grey line under the results: calls, tokens, cost and time."""
    display(HTML(f'<div style="margin:4px 0 10px;color:{MUTED};font-size:13px">{html.escape(text)}</div>'))


def reply_card(reply, label=""):
    """A help-desk reply in the 2.4 JSON format, as a card: the answer, then its labels."""
    if not reply:
        card(text_html("(The reply wasn't valid JSON.)"), label, ORANGE)
        return
    labels = (f'{badge(reply.get("category", "?"))} '
              f'<span style="color:{SOFT_INK};font-size:13px">confidence {reply.get("confidence", "?")}</span> '
              + (badge("needs a human", ORANGE) if reply.get("needs_human") else badge("no human needed", AQUA)))
    card(f'{text_html(reply.get("answer", ""))}<div style="margin-top:8px">{labels}</div>', label)


# How each part of a prompt is shown: the parts that come from elsewhere are highlighted.
CARRIED = {"ticket": "#ece9e1", "from step 1": "#d5f0e5", "from step 2": "#dbe8f8"}


def prompt_text(parts):
    """The prompt exactly as the model receives it: the parts joined together."""
    return "".join(part if isinstance(part, str) else part[1] for part in parts)


def prompt_html(parts):
    """The same prompt for the screen: the policies as one label, the carried-over parts highlighted."""
    pieces = []
    for part in parts:
        if isinstance(part, str):
            pieces.append(html.escape(part))
        elif part[0] in ("policies", "password policy", "VPN policy"):
            name = "the password and VPN policies" if part[0] == "policies" else f"the {part[0]}"
            pieces.append(f'<span style="background:#f1f0eb;border:1px solid #d6d4cc;border-radius:10px;'
                          f'padding:1px 8px;font-family:sans-serif;font-size:12px;color:{SOFT_INK}">'
                          f'📄 {name} · {len(part[1].split())} words</span>')
        else:
            kind, text = part
            pieces.append(f'<span style="background:{CARRIED[kind]};border-radius:4px;padding:1px 3px">'
                          f'<b style="font-family:sans-serif;font-size:11px;color:{SOFT_INK}">{kind} ▸ </b>'
                          f'{html.escape(text)}</span>')
    return (f'<div style="white-space:pre-wrap;font-family:monospace;font-size:12.5px;color:{INK}">'
            f'{"".join(pieces)}</div>')


def prompt_card(label, message, system=None):
    """What we send to the model: the system prompt (if any) and the message."""
    def heading(text):
        return f'<div style="font-size:11px;font-weight:700;color:{MUTED};margin:6px 0 2px">{text}</div>'
    body = (heading("SYSTEM PROMPT") + prompt_html(system) if system else "") + heading("MESSAGE") + prompt_html(message)
    card(body, label, MUTED)


print(f"Ready. Model: {MODEL}")

Ready. Model: qwen3:8b


In [3]:
# @title Answer caching { display-mode: "form" }
# @markdown When this is on, a request that is exactly the same as one already sent gets the saved answer
# @markdown back. Untick it (and run this step) to get fresh answers from Claude.
REPLAY = True  # @param {type:"boolean"}

import hashlib
import json
import os
from contextlib import contextmanager

import anthropic
from anthropic.resources.messages import Messages

REPLAY_DIR = "/content/claude_replay" if os.path.isdir("/content") else os.path.expanduser("~/claude_replay")
os.makedirs(REPLAY_DIR, exist_ok=True)
if not hasattr(Messages, "live_create"):
    Messages.live_create = Messages.create  # The real call, kept so we can switch back.
replay_count = {"saved answers reused": 0, "new calls": 0}


def replay_create(self, **request):
    """Every call to Claude comes through here: reuse a saved answer, or call Claude and save the answer."""
    # The connection type and address are part of the key, so a simulated outage never gets a saved answer,
    # and a local model never gets one of Claude's.
    connection = type(getattr(getattr(self._client, "_client", None), "_transport", None)).__name__
    address = str(self._client.base_url)
    key = hashlib.sha256(json.dumps([connection, address, request], sort_keys=True, default=str).encode()).hexdigest()
    path = os.path.join(REPLAY_DIR, f"{key}.json")
    if os.path.exists(path):
        replay_count["saved answers reused"] += 1
        with open(path) as file:
            return anthropic.types.Message.model_validate_json(file.read())
    response = Messages.live_create(self, **request)
    if isinstance(response, anthropic.types.Message):  # Streamed replies aren't saved.
        with open(path, "w") as file:
            file.write(response.model_dump_json())
    replay_count["new calls"] += 1
    return response


Messages.create = replay_create if REPLAY else Messages.live_create


@contextmanager
def fresh_answers():
    """Inside this block, every request goes to Claude, even with saved answers on. Used for timings."""
    current = Messages.create
    Messages.create = Messages.live_create
    try:
        yield
    finally:
        Messages.create = current


saved = len(os.listdir(REPLAY_DIR))
print(f"Saved answers {'on' if REPLAY else 'off'}. {saved} saved so far in {REPLAY_DIR}.")

Saved answers on. 10 saved so far in /content/claude_replay.


## The tickets and the policies
Five tickets from the help desk's queue, and two policies. The policies go into the prompt as plain
text.

In [4]:
# @title The tickets and the policies { display-mode: "form" }
TICKETS = [
    {"id": 1042, "title": "VPN disconnects every 10 minutes", "description": "NorthLink drops about every 10 minutes when I work from home, showing NGL-117. I am on Wi-Fi and on version 6.2. I turned off adapter power saving but it still happens. Logs are attached."},
    {"id": 1004, "title": "Account locked after password change", "description": "I changed my password this morning and now keep getting locked out every half hour."},
    {"id": 1050, "title": "Temporary password expired", "description": "The temporary password from my manager expired before I could use it, because my start was delayed."},
    {"id": 1012, "title": "Replace cracked laptop screen", "description": "Dropped my laptop on site. The screen is cracked but it still works with a monitor."},
    {"id": 1009, "title": "Access to finance shared folder", "description": "Please give me access to N:\\Finance\\Q3-Reports. I am covering for a colleague."},
]
TICKET = {ticket["id"]: ticket for ticket in TICKETS}

PASSWORD_POLICY = """# Password Reset Policy

This policy applies to all Northgate accounts. It replaces all earlier password guidance.

## Password rules

- Minimum length is 14 characters. We recommend a passphrase of several unrelated words.
- There are no required mixes of symbols or numbers.
- You cannot reuse any of your last 10 passwords.
- Never share your password, including with the help desk. We will never ask for it.

## How often you must change it

Passwords do not expire on a schedule. You do not need to change yours unless:

- you suspect someone else knows it
- IT Security asks you to, for example after a breach notice
- it appears on a list of compromised passwords, in which case you will be prompted at sign-in

## How to reset your password

Use the self-service page at password.northgate.example:

1. Enter your username.
2. Approve the request in the NorthKey Authenticator app.
3. Choose a new password that meets the rules above.

The new password takes effect within five minutes. Close and reopen Outlook and Teams if they keep asking you to sign in.

## If self-service does not work

If you lost your phone or cannot complete MFA, phone the help desk on extension 4357. The engineer will verify your identity using your employee number and your manager's name, then issue a temporary password valid for 24 hours.

If repeated failed attempts have locked your account, see the Account Lockout guide.

## If you think your password is compromised

Reset it straight away and report the event to the help desk as a security incident. If you typed your password into a suspicious page, phone us rather than opening a ticket, and see the Phishing Reporting guide."""

VPN_POLICY = """# VPN Troubleshooting and Error Codes

Try these steps first if NorthLink will not connect or keeps dropping:

1. Check that ordinary websites load. If they do not, the problem is your internet connection and not the VPN.
2. Quit NorthLink fully, then open it again.
3. Restart your laptop.
4. Check Company Apps for a NorthLink update. Versions older than 6.0 are not supported.

## Error codes

**NGL-809: certificate expired.** Your laptop's device certificate has expired, usually because the laptop has been away from the network for more than 60 days. You cannot fix this yourself. Connect to an office network and restart, or contact the help desk for a renewal.

**NGL-412: MFA timeout.** The approval request was not answered within 60 seconds. Connect again and approve the request promptly. Check your phone has a data connection.

**NGL-117: session dropped by client.** The connection was lost on your side. This commonly appears as repeated disconnects on Wi-Fi. Turn off the power-saving setting on your wireless adapter and update NorthLink. If the disconnects continue, open a ticket and attach the log from Help > Export Logs.

**NGL-530: route conflict.** Another network, often a home router or a second VPN, is using the same address range. Disconnect any other VPN. If you are at home, restart your router.

**NGL-221: account not permitted.** Your account is not in the VPN access group. This is common in a contractor's first week. Your manager must request access through the portal.

## Slow connections

All traffic passes through the tunnel, so speed depends on your own connection. Disconnect from the VPN for video calls. When working abroad, also read the Travel and Roaming guide.

## When to contact us

Open a ticket if a code above keeps returning or is not listed. Include the code, time and your location. If a whole office cannot connect, phone the help desk, because it is treated as a P1 incident."""

POLICIES = (
    f"<policy name='Password reset'>\n{PASSWORD_POLICY}\n</policy>\n\n"
    f"<policy name='VPN troubleshooting'>\n{VPN_POLICY}\n</policy>"
)


def ticket_text(ticket):
    return f"Ticket {ticket['id']}: {ticket['title']}\n{ticket['description']}"


rows = "".join(
    f'<tr><td style="padding:6px 14px 6px 0;color:{SOFT_INK};vertical-align:top">{ticket["id"]}</td>'
    f'<td style="padding:6px 14px 6px 0;color:{INK};font-weight:600;vertical-align:top">{html.escape(ticket["title"])}</td>'
    f'<td style="padding:6px 0;color:{INK}">{html.escape(ticket["description"])}</td></tr>'
    for ticket in TICKETS
)
display(HTML(f'<table style="border-collapse:collapse;background:#fff">{rows}</table>'))
caption(f"Policies: Password reset ({len(PASSWORD_POLICY.split())} words), "
        f"VPN troubleshooting ({len(VPN_POLICY.split())} words)")

1042,VPN disconnects every 10 minutes,"NorthLink drops about every 10 minutes when I work from home, showing NGL-117. I am on Wi-Fi and on version 6.2. I turned off adapter power saving but it still happens. Logs are attached."
1004,Account locked after password change,I changed my password this morning and now keep getting locked out every half hour.
1050,Temporary password expired,"The temporary password from my manager expired before I could use it, because my start was delayed."
1012,Replace cracked laptop screen,Dropped my laptop on site. The screen is cracked but it still works with a monitor.
1009,Access to finance shared folder,Please give me access to N:\Finance\Q3-Reports. I am covering for a colleague.


## Talking to the model, and counting the cost


In [5]:
# @title Prepare: send a request and count the cost { display-mode: "form" }
class Meter:
    """Adds up calls, tokens and cost for one pattern."""

    def __init__(self):
        self.lock = Lock()  # Step 4 runs several calls at the same time.
        self.calls = self.input_tokens = self.output_tokens = 0
        self.last = (0, 0)  # Tokens in and out of the latest call.

    def add(self, usage):
        with self.lock:
            self.calls += 1
            self.input_tokens += usage.input_tokens
            self.output_tokens += usage.output_tokens
            self.last = (usage.input_tokens, usage.output_tokens)

    @property
    def cost(self):
        return (self.input_tokens * PRICES[0] + self.output_tokens * PRICES[1]) / 1_000_000


def ask(prompt, meter, system=None, max_tokens=1000):
    """Send one prompt to the small model, record its usage, and return the text."""
    request = {
        "model": MODEL,
        "max_tokens": max_tokens,
        "thinking": {"type": "disabled"},
        "messages": [{"role": "user", "content": prompt}],
    }
    if system:
        request["system"] = system
    response = client.messages.create(**request)
    meter.add(response.usage)
    return "".join(block.text for block in response.content if block.type == "text").strip()


def read_json(text):
    """The JSON object inside a reply, or None if there isn't a valid one."""
    start, end = text.find("{"), text.rfind("}")
    try:
        return json.loads(text[start : end + 1])
    except ValueError:
        return None


SCORES = {}  # One row per pattern, filled in by each step, for the scorecard in step 5.


def record(pattern, meter, seconds, what_it_did):
    SCORES[pattern] = {
        "what it did": what_it_did,
        "calls": meter.calls,
        "tokens in": meter.input_tokens,
        "tokens out": meter.output_tokens,
        "cost": meter.cost,
        "seconds": round(seconds, 1),
    }
    caption(f"{pattern}: {meter.calls} call{'s' if meter.calls != 1 else ''} · "
            f"{meter.input_tokens:,} tokens in · {meter.output_tokens:,} out · ${meter.cost:.5f} · {seconds:.1f} s")


def last_call(meter):
    """Tokens in and out of the latest call, for the line under each step."""
    return f"{meter.last[0]:,} tokens in · {meter.last[1]:,} out"


print("Ready.")

Ready.


## 1. A single call
One well-written prompt answers the ticket, in the JSON format.
The grey card shows what we send: the system prompt and the ticket. The two policies are in the system
prompt; to keep it readable, they're shown as one label.

In [6]:
# @title Answer one ticket with one call { display-mode: "form" }
TICKET_ID = "1042"  # @param ["1042", "1004", "1050", "1012", "1009"]

SYSTEM_PARTS = [
    "You are the IT help-desk assistant for Northgate Logistics. Answer from these\n"
    "policies where they apply, and say so when they don't cover the question.\n\n",
    ("policies", POLICIES),
    "\n\nReturn only a JSON object with these keys: answer (the reply to the employee), category (one of\n"
    "network, account_access, hardware, software, other), confidence (0 to 1), needs_human (true or false).",
]
HELPDESK_SYSTEM = prompt_text(SYSTEM_PARTS)

ticket = TICKET[int(TICKET_ID)]
message = [("ticket", ticket_text(ticket))]
meter = Meter()
start = time.perf_counter()
reply = ask(prompt_text(message), meter, system=HELPDESK_SYSTEM)
seconds = time.perf_counter() - start

prompt_card("what we send", message, system=SYSTEM_PARTS)
reply_card(read_json(reply), "the reply")
record("1. Single call", meter, seconds, "answered 1 ticket · policies sent once")

## 2. A prompt chain
Three fixed steps, each one a smaller, easier job: **classify** the ticket, **draft** the reply, then
**check** the draft against the policy. Our code decides the order and passes each output on.
Each step shows what we send and the answer. The highlighted parts are carried over: the category from
step 1 goes into step 2's prompt, and the draft from step 2 goes into step 3's.
**Watch for:** each intermediate output, and the check catching (or passing) the draft.

The check is a new call to the same small model, with a narrower job: it only compares the draft with the
policies. It sees the draft fresh, like a reviewer, without the reasoning that produced it. It can share
the writer's blind spots, which is why a bigger or different model is sometimes the checker.

In [7]:
# @title Classify, then draft, then check { display-mode: "form" }
TICKET_ID = "1004"  # @param ["1042", "1004", "1050", "1012", "1009"]

ticket = TICKET[int(TICKET_ID)]
meter = Meter()
start = time.perf_counter()
# Step 1: classify the ticket.
classify = [
    ("ticket", ticket_text(ticket)),
    "\n\nClassify this ticket. Reply with one word: network, account_access, hardware, software or other.",
]
category = ask(prompt_text(classify), meter, max_tokens=10)
prompt_card("step 1 · classify · what we send", classify)
card(badge(category), f"step 1 · classify · the answer ({last_call(meter)})")

# Step 2: draft a reply, using the category from step 1.
write = [
    ("ticket", ticket_text(ticket)),
    "\n\nCategory: ",
    ("from step 1", category),
    "\n\nWrite a short, friendly reply to the employee. Use these policies where they apply.\n\n",
    ("policies", POLICIES),
]
draft = ask(prompt_text(write), meter, max_tokens=600)
prompt_card("step 2 · draft · what we send", write)
card(text_html(draft), f"step 2 · draft · the answer ({last_call(meter)})")

# Step 3: check the draft from step 2 against the policies.
review = [
    'Check this reply against the policies. Return only JSON: {"follows_policy": true or false, '
    '"problems": ["..."]}.\n\n<reply>\n',
    ("from step 2", draft),
    "\n</reply>\n\n",
    ("policies", POLICIES),
]
check = ask(prompt_text(review), meter, max_tokens=400)
prompt_card("step 3 · check · what we send", review)
result = read_json(check) or {}
problems = "".join(f"<li>{html.escape(str(problem))}</li>" for problem in result.get("problems") or [])
card(f'{verdict(result.get("follows_policy"))} follows the policies'
     + (f'<ul style="margin:6px 0 0;color:{INK}">{problems}</ul>' if problems else ""),
     f"step 3 · check · the answer ({last_call(meter)})", AQUA if result.get("follows_policy") else ORANGE)

record("2. Prompt chain", meter, time.perf_counter() - start, "answered 1 ticket in 3 steps · policies sent twice")

## 3. A router
The small model sorts each ticket into **password**, **hardware** or **other**. Each route has its own
system prompt, written for that kind of request: different instructions, and a different policy.
The router's prompt and the three system prompts are shown first; then each ticket, the route the router
chose, and the answer.

In [8]:
# @title Sort three tickets into routes, then answer each { display-mode: "form" }
ROUTE_PARTS = {  # One system prompt per route: different instructions, and a different policy.
    "password": [
        "You handle password and sign-in problems for Northgate Logistics. Follow the password "
        "policy exactly, and never ask for a password.\n\n<policy>\n",
        ("password policy", PASSWORD_POLICY),
        "\n</policy>",
    ],
    "hardware": [
        "You handle hardware problems for Northgate Logistics. Say whether the device needs a "
        "repair or a replacement, and what the employee should do until then.",
    ],
    "other": [
        "You are the general IT help desk for Northgate Logistics. Answer briefly. If the ticket "
        "is about the VPN, use this guide.\n\n<policy>\n",
        ("VPN policy", VPN_POLICY),
        "\n</policy>",
    ],
}
ROUTE_PROMPTS = {route: prompt_text(parts) for route, parts in ROUTE_PARTS.items()}
ROUTE_COLOURS = {"password": BLUE, "hardware": ORANGE, "other": MUTED}


def router_message(ticket):
    """What the router reads: the ticket, and one question with a one-word answer."""
    return [
        ("ticket", ticket_text(ticket)),
        "\n\nIs this ticket about a password, about hardware, or other? "
        "Reply with one word: password, hardware or other.",
    ]


TICKETS_TO_ROUTE = (TICKET[1050], TICKET[1012], TICKET[1042])

# What stays the same for every ticket: the router's prompt, and the three system prompts.
prompt_card(f"the router · what we send (the same for every ticket; here ticket {TICKETS_TO_ROUTE[0]['id']})",
            router_message(TICKETS_TO_ROUTE[0]))
rows = "".join(
    f'<tr><td style="padding:8px 12px 8px 0;vertical-align:top">{badge(route, ROUTE_COLOURS[route])}</td>'
    f'<td style="padding:8px 0;border-top:1px solid #e1e0d9">{prompt_html(parts)}</td></tr>'
    for route, parts in ROUTE_PARTS.items()
)
card(f'<table style="border-collapse:collapse;width:100%">{rows}</table>', "the three system prompts, one per route", MUTED)

meter = Meter()
start = time.perf_counter()
for ticket in TICKETS_TO_ROUTE:
    route = ask(prompt_text(router_message(ticket)), meter, max_tokens=5).lower().strip(" .")
    route = route if route in ROUTE_PROMPTS else "other"
    route_tokens = last_call(meter)
    answer = ask(ticket_text(ticket), meter, system=ROUTE_PROMPTS[route], max_tokens=500)
    card(f'<div style="margin-bottom:6px;color:{SOFT_INK};font-size:13px">the router says '
         f'{badge(route, ROUTE_COLOURS[route])} <span style="color:{MUTED}">({route_tokens})</span>'
         f' → answered with the <b>{route}</b> system prompt</div>'
         f'{text_html(answer)}<div style="margin-top:6px;color:{MUTED};font-size:12px">{last_call(meter)}</div>',
         f"ticket {ticket['id']} · {html.escape(ticket['title'])}", ROUTE_COLOURS[route])

record("3. Router", meter, time.perf_counter() - start, "answered 3 tickets · one policy per route, or none")

password,"You handle password and sign-in problems for Northgate Logistics. Follow the password policy exactly, and never ask for a password. <policy> 📄 the password policy · 290 words </policy>"
hardware,"You handle hardware problems for Northgate Logistics. Say whether the device needs a repair or a replacement, and what the employee should do until then."
other,"You are the general IT help desk for Northgate Logistics. Answer briefly. If the ticket is about the VPN, use this guide. <policy> 📄 the VPN policy · 322 words </policy>"


## 4. Parallel calls
Three independent checks on one draft reply: **tone**, **policy compliance** and **does it need a human?**
They don't depend on each other, so they can run at the same time. We run them one after the other,
then all at once, and combine the three verdicts.

Each check answers pass or fail, with a reason. Our code then combines them with one rule: send the reply
only if all three pass.

The draft below has a deliberate mistake: it tells the employee their password expires every
90 days, which the policy says isn't true.

In [9]:
# The draft reply the three checks look at.
DRAFT = """Hi! Sorry about the lockouts. They usually come from a device still using your old password,
so sign out of Outlook and Teams on your phone and sign back in. Remember that passwords expire every
90 days, so you'll need to change it again soon. If it keeps happening, reply and we'll look into it."""

In [10]:
# @title Run the three checks, one after the other and then all at once { display-mode: "form" }
CHECKS = {
    "tone": "Is this reply polite, clear and short enough for an employee? ",
    "policy": f"Does this reply follow these policies, with nothing that contradicts them?\n\n{POLICIES}\n\n",
    "needs_human": "Can this reply be sent as it is, without an engineer looking at the case first? ",
}
CHECK_NAMES = {"tone": "Tone is right", "policy": "Follows the policy", "needs_human": "No engineer needed"}


def run_check(name, meter):
    reply = ask(
        f'{CHECKS[name]}Return only JSON: {{"pass": true or false, "reason": "one sentence"}}.\n\n'
        f"<reply>\n{DRAFT}\n</reply>",
        meter, max_tokens=200,
    )
    return name, read_json(reply) or {"pass": False, "reason": reply}


with fresh_answers():  # Timings need real calls, so saved answers are skipped in this step.
    start = time.perf_counter()
    sequential = [run_check(name, Meter()) for name in CHECKS]
    sequential_seconds = time.perf_counter() - start

    meter = Meter()
    start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=len(CHECKS)) as pool:  # All three checks at the same time.
        verdicts = dict(pool.map(lambda name: run_check(name, meter), CHECKS))
    parallel_seconds = time.perf_counter() - start

rows = "".join(
    f'<tr><td style="padding:6px 14px 6px 0;color:{INK};font-weight:600">{CHECK_NAMES[name]}</td>'
    f'<td style="padding:6px 14px 6px 0">{verdict(result.get("pass"))}</td>'
    f'<td style="padding:6px 0;color:{INK}">{html.escape(str(result.get("reason")))}</td></tr>'
    for name, result in verdicts.items()
)
failed = [CHECK_NAMES[name] for name, result in verdicts.items() if not result.get("pass")]
combined = (f'{badge("send it", AQUA)} <span style="color:{INK}">all three checks passed</span>' if not failed
            else f'{badge("hold it", ORANGE)} <span style="color:{INK}">{len(failed)} of {len(CHECKS)} checks failed: '
                 f'{html.escape(", ".join(failed))}</span>')
card(f'<table style="border-collapse:collapse">{rows}</table>'
     f'<div style="margin-top:10px;color:{SOFT_INK}">Our rule: send the reply only if all three checks pass.</div>'
     f'<div style="margin-top:4px;color:{SOFT_INK}">Result: {combined}</div>',
     "the three verdicts", AQUA if not failed else ORANGE)

# The two timings as bars, so the difference is visible at a glance.
longest = max(sequential_seconds, parallel_seconds)
bars = "".join(
    f'<div style="display:flex;align-items:center;margin:4px 0">'
    f'<div style="width:170px;color:{SOFT_INK}">{label}</div>'
    f'<div style="width:{max(2, 300 * seconds / longest):.0f}px;height:14px;background:{colour};border-radius:3px"></div>'
    f'<div style="margin-left:8px;color:{INK};font-weight:600">{seconds:.1f} s</div></div>'
    for label, seconds, colour in (("one after the other", sequential_seconds, MUTED),
                                   ("all at once", parallel_seconds, BLUE))
)
card(bars, "time for the three checks")

record("4. Parallel calls", meter, parallel_seconds, "checked a draft we wrote, no answer · policies sent once")
caption("The run one after the other made 3 more calls, only to compare the time; they're not counted above.")

Tone is right,pass,"The reply is polite, clear, and concise enough for an employee."
Follows the policy,pass,The reply does not contradict the policies and follows the guidelines provided.
No engineer needed,pass,The reply addresses the issue with a common solution and provides next steps without requiring an engineer's immediate involvement.


## 5. Scorecard


In [11]:
# @title Compare the patterns { display-mode: "form" }
scorecard = pd.DataFrame(SCORES).T
display(
    scorecard.style.format({"calls": "{:.0f}", "tokens in": "{:,.0f}", "tokens out": "{:,.0f}",
                            "cost": "${:.5f}", "seconds": "{:.1f} s"})
    .set_properties(subset=["what it did"], **{"text-align": "left"})
)

,what it did,calls,tokens in,tokens out,cost,seconds
1. Single call,answered 1 ticket · policies sent once,1,984,126,$0.00000,133.8 s
2. Prompt chain,answered 1 ticket in 3 steps · policies sent twice,3,"2,027",238,$0.00000,8.2 s
3. Router,"answered 3 tickets · one policy per route, or none",6,"1,296",419,$0.00000,12.3 s
4. Parallel calls,"checked a draft we wrote, no answer · policies sent once",3,3,81,$0.00000,1.5 s
